In [7]:
import hashlib
import os
import platform
import uuid
import re
from bs4 import BeautifulSoup
import requests

STUDENT_ID = 20244080122
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")


[Conda 环境] None
[学号] 20244080122
[机器指纹] b1eeb302b80bf458
[RUN_ID] 20244080122-b1eeb302b80bf458


In [8]:
# 1.检查字符串仅包含大小写字母、数字 a‑z A‑Z 0‑9
def check_alnum(s):
    return bool(re.fullmatch(r"[A-Za-z0-9]+", s))

# 2.匹配a后面跟0个或多个b
def match_a_zeroormore_b(s):
    return bool(re.fullmatch(r"ab*", s))

#3.匹配a后面跟1个或多个b
def match_a_oneormore_b(s):
    return bool(re.fullmatch(r"ab+", s))

#4.以a开头，以b结尾，中间任意字符
def match_start_a_end_b(s):
    return bool(re.fullmatch(r"a.*b", s))

#5.删除IP各段前导零，全零保留一个0
def ip_strip_leading_zero(ip_str):
    def replace_octet(m):
        num = m.group(1)
        return str(int(num))
    return re.sub(r"\b0*(\d+)\b", replace_octet, ip_str)

#6.空格↔下划线互换
def swap_space_underscore(s):
    # 先把空格临时替换标记，再互换
    s1 = re.sub(r" ", "\x01", s)
    s2 = re.sub(r"_", " ", s1)
    s3 = re.sub(r"\x01", "_", s2)
    return s3

#7. yyyy‑mm‑dd → dd‑mm‑yyyy
def date_ymd_to_dmy(s):
    return re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3-\2-\1", s)

#8.找出所有a或e开头的单词
def find_words_start_a_e(s):
    return re.findall(r"\b[aeAE]\w*\b", s)

#9. Road缩写为Rd
def road_to_rd(s):
    return re.sub(r"\bRoad\b", "Rd", s)

#10.找出长度至少4字符的单词
def find_word_ge4(s):
    return re.findall(r"\b\w{4,}\b", s)

#11.只保留字母数字，删除其余字符
def keep_alnum_only(s):
    return re.sub(r"[^A-Za-z0-9]", "", s)

#12.删除括号以及括号内部内容
def remove_parentheses_content(s):
    return re.sub(r"\([^)]*\)", "", s).strip()


# ========== 测试用例 ==========
print("1 check_alnum('Abc123'):", check_alnum("Abc123"), " | check_alnum('Abc!'):", check_alnum("Abc!"))
print("2 match_a_zeroormore_b('a'):",match_a_zeroormore_b('a')," | match_a_zeroormore_b('abbb'):",match_a_zeroormore_b('abbb'))
print("3 match_a_oneormore_b('a'):",match_a_oneormore_b('a')," | match_a_oneormore_b('ab'):",match_a_oneormore_b('ab'))
print("4 match_start_a_end_b('aXXb'):",match_start_a_end_b('aXXb')," | match_start_a_end_b('aXXc'):",match_start_a_end_b('aXXc'))
print("5 ip_strip_leading_zero('192.168.001.010') →", ip_strip_leading_zero("192.168.001.010"))
print("6 swap_space_underscore('hello world_test') →", swap_space_underscore("hello world_test"))
print("7 date_ymd_to_dmy('2025‑10‑05') →", date_ymd_to_dmy("2025-10-05"))
print("8 find_words_start_a_e('Apple egg cat End') →", find_words_start_a_e("Apple egg cat End"))
print("9 road_to_rd('Main Road') →", road_to_rd("Main Road"))
print("10 find_word_ge4('hi test apple cat') →", find_word_ge4("hi test apple cat"))
print("11 keep_alnum_only('abc!123#xy') →", keep_alnum_only("abc!123#xy"))
print("12 remove_parentheses_content测试：")
for t in ["example (.com)", "w3resource", "github (.com) stackoverflow (.com)"]:
    print(f" input:{t:45} output:{remove_parentheses_content(t)}")


1 check_alnum('Abc123'): True  | check_alnum('Abc!'): False
2 match_a_zeroormore_b('a'): True  | match_a_zeroormore_b('abbb'): True
3 match_a_oneormore_b('a'): False  | match_a_oneormore_b('ab'): True
4 match_start_a_end_b('aXXb'): True  | match_start_a_end_b('aXXc'): False
5 ip_strip_leading_zero('192.168.001.010') → 192.168.1.10
6 swap_space_underscore('hello world_test') → hello_world test
7 date_ymd_to_dmy('2025‑10‑05') → 05-10-2025
8 find_words_start_a_e('Apple egg cat End') → ['Apple', 'egg', 'End']
9 road_to_rd('Main Road') → Main Rd
10 find_word_ge4('hi test apple cat') → ['test', 'apple']
11 keep_alnum_only('abc!123#xy') → abc123xy
12 remove_parentheses_content测试：
 input:example (.com)                                output:example
 input:w3resource                                    output:w3resource
 input:github (.com) stackoverflow (.com)            output:github  stackoverflow


In [9]:
def retrieve_climate_table(soup):
    for table in soup.find_all("table"):
        cap = table.find("caption")
        cap_text = ""
        if cap:
            cap_text = cap.get_text(strip=True)
        if cap_text.startswith("Climate data for"):
            return table
        first_th = table.find("th")
        if first_th:
            thtxt = first_th.get_text(strip=True)
            if thtxt.startswith("Climate data for"):
                return table
    return None

# 本地模拟HTML，不用访问外网
sample_html = """
<table>
<caption>Climate data for Madison, Wisconsin</caption>
<tbody>
<tr><th>Month</th><td>Jan</td><td>Feb</td></tr>
<tr><th>Average high °C</th><td>-2</td><td>0</td></tr>
<tr><th>Annual precipitation mm</th><td>920</td></tr>
<tr><th>Source</th><td>NOAA</td></tr>
</tbody>
</table>
"""
soup = BeautifulSoup(sample_html, "html.parser")
tbl1 = retrieve_climate_table(soup)
print(f"Madison 找到气候表? {tbl1 is not None}")

if tbl1:
    print("\nMadison表格直接子节点标签：")
    for child in tbl1.children:
        if hasattr(child,"name"):
            print(" -", child.name)


Madison 找到气候表? True

Madison表格直接子节点标签：
 - None
 - caption
 - None
 - tbody
 - None


In [10]:
def list_climate_table_row_names(soup):
    table_tag = retrieve_climate_table(soup)
    if table_tag is None:
        return None
    rows = table_tag.find_all("tr")
    row_names = []
    for tr in rows:
        ths = tr.find_all("th")
        if len(ths) == 0:
            continue
        first_th = ths[0]
        txt = first_th.get_text(strip=True)
        skip_keywords = ["Climate data for", "Jan", "February", "Mar", "Apr", "May",
                         "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Source", "Month"]
        if any(k in txt for k in skip_keywords):
            continue
        row_names.append(txt)
    return row_names

rowlist = list_climate_table_row_names(soup)
print("\n=== list_climate_table_row_names 测试 ===")
print("Madison气候表行名：")
print(rowlist)



=== list_climate_table_row_names 测试 ===
Madison气候表行名：
['Average high °C', 'Annual precipitation mm']


In [11]:
def list_climate_table_row_names(soup):
    table_tag = retrieve_climate_table(soup)
    if table_tag is None:
        return None
    rows = table_tag.find_all("tr")
    row_names = []
    for tr in rows:
        ths = tr.find_all("th")
        if len(ths) == 0:
            continue
        first_th = ths[0]
        txt = first_th.get_text(strip=True)
        skip_keywords = ["Climate data for", "Jan", "February", "Mar", "Apr", "May",
                         "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Source", "Month"]
        if any(k in txt for k in skip_keywords):
            continue
        row_names.append(txt)
    return row_names

rowlist = list_climate_table_row_names(soup)
print("=== list_climate_table_row_names 测试 ===")
print("Madison气候表行名：")
print(rowlist)


=== list_climate_table_row_names 测试 ===
Madison气候表行名：
['Average high °C', 'Annual precipitation mm']


使用模拟HTML替代维基百科网页，规避外网访问超时问题。
表格节点结构说明：
气候数据 <table> 的直接子节点为 <caption>、<tbody>；
<caption>存放表格标题；
<tbody>包含全部<tr>数据行，每行包含<th>行标签与<td>数据单元格。
过滤规则：跳过月份、表头、来源注释行，只保留指标行名称。
